# Gradient boosting

Data Scientist track, lesson DS8. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** see boosting as small trees that each fix the errors of the ones before, stop a boosted model early, tune its learning rate and tree size with cross-validation, and compare it fairly with a forest and a logistic model.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The split, the boosted model and the forest

The columns, the split of lessons DS2 and DS4 (the test part stays untouched until the last unit), the five folds of lesson DS5, the day-7 model, the forest of lesson DS7 and a gradient-boosted model. A tree needs no scaling, so the number columns go in as they are. Early stopping is off in `boost` unless you switch it on. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)       # 13,362 and 4,454 people
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)   # 10,021 and 3,341, cut from the training part
folds = StratifiedKFold(5, shuffle=True, random_state=SEED)       # the five folds of lesson DS5


def logit():
    """The day-7 model of lesson DS2: one-hot text columns, scaled number columns, logistic regression."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))


def auc_gain(p_new, p_base, y_true, resamples=2000):
    """ROC AUC of one model minus another on the same people, with a 95 percent bootstrap interval; both models are scored on the same resamples."""
    rng = np.random.default_rng(SEED)
    y_true, a, b = np.asarray(y_true), np.asarray(p_new), np.asarray(p_base)
    gains = []
    for _ in range(resamples):
        i = rng.integers(0, len(y_true), len(y_true))
        gains.append(roc_auc_score(y_true[i], a[i]) - roc_auc_score(y_true[i], b[i]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return [round(float(roc_auc_score(y_true, a) - roc_auc_score(y_true, b)), 3) + 0.0, round(float(low), 3) + 0.0, round(float(high), 3) + 0.0]


def prep():
    """One-hot text columns and the number columns as they are: trees do not care about units."""
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", "passthrough", WEEK1)], sparse_threshold=0)


def boost(**settings):
    """A gradient-boosted model of small trees. Early stopping is off unless you pass early_stopping=True."""
    settings.setdefault("early_stopping", False)
    return make_pipeline(prep(), HistGradientBoostingClassifier(random_state=SEED, **settings))


def forest(trees=300, leaf=100):
    """The random forest of lesson DS7."""
    return make_pipeline(prep(), RandomForestClassifier(n_estimators=trees, min_samples_leaf=leaf, n_jobs=-1, random_state=SEED))


def staged_auc(model, X_part, y_part):
    """ROC AUC after 1, 2, 3, ... trees of a fitted boosted model."""
    Z = model[0].transform(X_part)
    return [float(roc_auc_score(y_part, p[:, 1])) for p in model[-1].staged_predict_proba(Z)]


print(len(X_fit), len(X_val), len(X_test))

## Unit 1: Trees in sequence

**Predict first.** With the default settings (trees of up to 31 leaves, a learning rate of 0.1) the AUC on the fitting rows keeps rising. At about which tree will the AUC on the validation rows be highest: about the 6th, about the 60th or about the 300th?

In [ ]:
SMALL = dict(learning_rate=0.05, max_leaf_nodes=4, min_samples_leaf=100)           # many small trees: each one has at most 4 leaves
stages = {}
for tag, settings in {"defaults": {}, "small trees": SMALL}.items():
    m = boost(max_iter=300, **settings).fit(X_fit, y_fit)                          # 300 trees, one after another
    stages[tag] = (staged_auc(m, X_fit, y_fit), staged_auc(m, X_val, y_val))
rows = [(k, *[round(stages[t][j][k - 1], 3) for t in stages for j in (0, 1)]) for k in [1, 5, 10, 25, 50, 100, 300]]
print(pd.DataFrame(rows, columns=["trees", "defaults: fitting", "defaults: validation", "small: fitting", "small: validation"]).to_string(index=False))
print({t: (int(np.argmax(stages[t][1])) + 1, round(max(stages[t][1]), 3)) for t in stages})       # the number of trees with the best validation AUC, and that AUC

## Unit 2: Early stopping

**Predict first.** Will early stopping cost AUC compared with all 300 trees, and about how many trees will it keep: about 30, about 100 or all 300?

In [ ]:
full = boost(max_iter=300).fit(X_fit, y_fit)                                                  # the default settings, all 300 trees
stopped = boost(max_iter=300, early_stopping=True, validation_fraction=0.15, n_iter_no_change=10).fit(X_fit, y_fit)
auc_full, auc_stopped = (roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]) for m in (full, stopped))
print(300, round(float(auc_full), 3), stopped[-1].n_iter_, round(float(auc_stopped), 3))
stops = []
for seed in range(100, 110):                                                                  # ten other seeds: the held-back 15 percent changes
    m = make_pipeline(prep(), HistGradientBoostingClassifier(max_iter=300, early_stopping=True, validation_fraction=0.15, n_iter_no_change=10, random_state=seed)).fit(X_fit, y_fit)
    stops.append(int(m[-1].n_iter_))
print(stops)

## Unit 3: The learning rate sets how many trees

**Predict first.** With early stopping choosing the number of trees, will a learning rate of 0.01 beat 0.1 on the validation people by more than 0.005 AUC, and about how many times as many trees will it keep?

In [ ]:
lr_rows = []
for lr in [0.3, 0.1, 0.03, 0.01]:
    m = boost(learning_rate=lr, max_leaf_nodes=8, min_samples_leaf=100, max_iter=2000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=20).fit(X_fit, y_fit)
    lr_rows.append((lr, int(m[-1].n_iter_), round(float(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])), 3)))
print(pd.DataFrame(lr_rows, columns=["learning_rate", "trees kept", "validation AUC"]).to_string(index=False))

## Unit 4: Tune the rest on the folds

**Predict first.** How far apart will the best five settings score on the folds: under 0.003, about 0.01 or over 0.02? And will the best beat the day-7 model's mean on the same folds?

In [ ]:
space = {"histgradientboostingclassifier__learning_rate": [0.03, 0.1],
         "histgradientboostingclassifier__max_leaf_nodes": [4, 8, 16, 31],
         "histgradientboostingclassifier__min_samples_leaf": [20, 100, 300]}
search = RandomizedSearchCV(boost(max_iter=1000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=20), space, n_iter=12, cv=folds, scoring="roc_auc", random_state=SEED, n_jobs=-1).fit(X_train, y_train)
res = pd.DataFrame(search.cv_results_)
splits = res[[f"split{k}_test_score" for k in range(5)]]
res["low"], res["high"] = splits.min(axis=1), splits.max(axis=1)
for _, r in res.sort_values("mean_test_score", ascending=False).head(5).iterrows():
    print(round(float(r["mean_test_score"]), 4), round(float(r["low"]), 3), round(float(r["high"]), 3), {k.split("__")[1]: v for k, v in r["params"].items()})
print("worst", round(float(res["mean_test_score"].min()), 4), "day-7 model on the same folds", round(float(cross_val_score(logit(), X_train, y_train, cv=folds, scoring="roc_auc").mean()), 4))

## Unit 5: The verdict on the test part

**Predict first.** Will the boosted model beat the day-7 model, or the forest, by more than 0.005 AUC?

In [ ]:
p_day7 = logit().fit(X_train, y_train).predict_proba(X_test)[:, 1]              # the test part is used once, here
p_boost = search.best_estimator_.predict_proba(X_test)[:, 1]
p_forest = forest().fit(X_train, y_train).predict_proba(X_test)[:, 1]
print("day-7 model", round(float(roc_auc_score(y_test, p_day7)), 3), "boosted", round(float(roc_auc_score(y_test, p_boost)), 3), "forest", round(float(roc_auc_score(y_test, p_forest)), 3))
print("boosted over day-7", auc_gain(p_boost, p_day7, y_test), "boosted over forest", auc_gain(p_boost, p_forest, y_test))

## Finish it

### Exercise 1: Finish it

**Task:** **Bigger trees, same count.** Fit a boosted model of 300 trees with a learning rate of 0.05 and at least 100 people per leaf on the fitting part, for each number of leaves in `leaves` (the loop is given: 2, 8 and 31), and put the validation AUC after all 300 trees, for each, in `answer_1`, in the order of the list, each rounded to 3 decimals. Use `staged_auc`.

**Expected:** `answer_1` is a list of three numbers.

In [ ]:
after_300 = []
for leaves in [2, 8, 31]:                   # given: trees of 2, 8 and 31 leaves
    # fit boost(max_iter=300, learning_rate=0.05, max_leaf_nodes=leaves, min_samples_leaf=100) on the fitting part and keep its validation AUC after all 300 trees
    pass
answer_1 = None  # replace None with the three validation AUCs in the order of the list, each rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [0.701, 0.694, 0.682], traps=[([0.701, 0.701, 0.693], "That is the best score over the 300 trees, not the score after all 300. Take the last number of staged_auc.")], tol=5e-4)

<details>
<summary>Hint</summary>

`boost(max_iter=300, learning_rate=0.05, max_leaf_nodes=leaves, min_samples_leaf=100).fit(X_fit, y_fit)` builds the model; `staged_auc(m, X_val, y_val)` gives a list of 300 scores, and its last item is the score after 300 trees.

</details>

<details>
<summary>Half-solution</summary>

```python
m = boost(max_iter=300, learning_rate=0.05, max_leaf_nodes=leaves, min_samples_leaf=100).fit(X_fit, y_fit)
after_300.append(round(staged_auc(m, X_val, y_val)[-1], 3))
```

</details>

<details>
<summary>Solution</summary>

```python
after_300 = []
for leaves in [2, 8, 31]:
    m = boost(max_iter=300, learning_rate=0.05, max_leaf_nodes=leaves, min_samples_leaf=100).fit(X_fit, y_fit)
    after_300.append(round(staged_auc(m, X_val, y_val)[-1], 3))
answer_1 = after_300
```

The scores after 300 trees are 0.701, 0.694, 0.682 for 2, 8 and 31 leaves. The bigger the trees, the earlier the model memorizes: trees of two leaves are still improving or flat at tree 300, and trees of 31 leaves have long passed their best.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Pick the learning rate on folds.** Before you start, say in one sentence what the result looks like: a rate from the list and a mean AUC near 0.69. For each learning rate in 0.3, 0.1 and 0.03, score a boosted model with 8 leaves per tree, at least 100 people per leaf, up to 2,000 trees and early stopping (`validation_fraction=0.15`, `n_iter_no_change=20`) on the five folds of the training part with ROC AUC. Put the list [the rate with the best mean, that mean rounded to 3 decimals] in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.03, 0.691], traps=[([0.1, 0.697], "That is the rate that wins on the validation part in unit 3. Score on the five folds of the training part.")], tol=5e-4)

<details>
<summary>Hint</summary>

`cross_val_score(boost(learning_rate=lr, max_leaf_nodes=8, min_samples_leaf=100, max_iter=2000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=20), X_train, y_train, cv=folds, scoring="roc_auc")` gives the five fold scores of one rate.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A rate of 0.1 or lower and a mean a hair under 0.692, the same level as the day-7 model."*

```python
five_rows = []
for lr in [0.3, 0.1, 0.03]:
    s = cross_val_score(boost(learning_rate=lr, ...), X_train, y_train, cv=folds, scoring="roc_auc")
    five_rows.append((lr, round(float(s.mean()), 3)))
```

</details>

<details>
<summary>Solution</summary>

```python
five_rows = []
for lr in [0.3, 0.1, 0.03]:
    s = cross_val_score(boost(learning_rate=lr, max_leaf_nodes=8, min_samples_leaf=100, max_iter=2000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=20),
                        X_train, y_train, cv=folds, scoring="roc_auc")
    five_rows.append((lr, round(float(s.mean()), 3)))
answer_2 = [max(five_rows, key=lambda r: r[1])[0], max(r[1] for r in five_rows)]
```

The means are 0.3: 0.686, 0.1: 0.689, 0.03: 0.691, so the best rate is 0.03. The means rise as the rate falls, by 0.005 from the biggest rate to the smallest, a quarter of the range of the folds of any one rate: a small rate is a safe choice and not a large gain.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague fits `HistGradientBoostingClassifier` with every setting at its default on the whole fitting part and on a pilot of 5,000 people (the code is given), reads the two validation AUCs, and writes that 5,000 people instead of 10,021 cost the model the gap between the two scores. Print `n_iter_` of both models first. Then change the last line: refit the pilot model with `early_stopping=True` and put its validation AUC in `answer_3`, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0.5 and 1.

In [ ]:
pilot = X_fit.sample(5000, random_state=5)               # given: a pilot of 5,000 people
y_pilot = y_fit.loc[pilot.index]
rows_ = []
for X_part, y_part in [(X_fit, y_fit), (pilot, y_pilot)]:
    m = make_pipeline(prep(), HistGradientBoostingClassifier(random_state=SEED)).fit(X_part, y_part)           # every setting at its default
    rows_.append((int(m[-1].n_iter_), round(float(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])), 3)))
print(rows_)                  # for the whole fitting part and for the pilot: trees built, validation AUC
answer_3 = rows_[1][1]        # colleague: "5,000 people instead of 10,021 cost this much AUC (see the gap between the two scores)"
# change the last line: refit the pilot model with early_stopping=True and put its validation AUC in answer_3, rounded to 3 decimals

In [ ]:
_ = check("Exercise 3", "answer_3", 0.681, traps={0.669: "That is the pilot with the defaults, which built 100 trees without stopping. Switch early stopping on for the pilot."}, tol=5e-4)

<details>
<summary>Hint</summary>

The two models were not fitted the same way. Look at how many trees each one built, and read what the documentation says about `early_stopping='auto'`.

</details>

<details>
<summary>Half-solution</summary>

```python
pilot_stop = make_pipeline(prep(), HistGradientBoostingClassifier(random_state=SEED, early_stopping=True)).fit(pilot, y_pilot)
answer_3 = round(float(roc_auc_score(y_val, pilot_stop.predict_proba(X_val)[:, 1])), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
pilot_stop = make_pipeline(prep(), HistGradientBoostingClassifier(random_state=SEED, early_stopping=True)).fit(pilot, y_pilot)
answer_3 = round(float(roc_auc_score(y_val, pilot_stop.predict_proba(X_val)[:, 1])), 3)
```

The default `early_stopping="auto"` switches early stopping on when the training part holds more than 10,000 people, and off below that. The whole fitting part has 10,021 people, so early stopping switched itself on and the model built 34 trees; the pilot has 5,000, so it built all 100 and memorized. The colleague's gap is 0.021 AUC. With early stopping on for both, the pilot scores 0.681 and the gap is 0.009: most of the "cost of less data" came from a setting that changed with the number of rows. Print `n_iter_` whenever a boosted model's default behavior matters.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")